# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'data.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: data.csv
Ukuran file: 737.9 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [2]:
lazy_data = pl.scan_csv(DATA_PATH) if DATA_PATH.suffix == '.csv' else pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'id': Int64, 'source': String, 'title': String, 'image': String, 'url': String, 'content': String, 'date': String, 'embedding': String, 'created_at': String, 'updated_at': String, 'summary': String})


jumlah_baris,id_null,source_null,title_null,image_null,url_null,content_null,date_null,embedding_null,created_at_null,updated_at_null,summary_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
32735,0,0,0,0,0,0,0,0,0,0,10


In [3]:
contoh_data = lazy_data.head(10).collect()
contoh_data

id,source,title,image,url,content,date,embedding,created_at,updated_at,summary
i64,str,str,str,str,str,str,str,str,str,str
83,"""tempo""","""Depo Plumpang Terbakar, Anggot…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Anggota Ko…","""2023-03-04 06:18:13+00""","""[-0.01590039,-0.034130897,0.00…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Anggota Komisi VII DPR RI Rofi…"
84,"""tempo""","""Jokowi Perintahkan Wapres Ma'r…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Presiden J…","""2023-03-04 06:04:38+00""","""[-0.017608976,-0.021786924,0.0…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Presiden Joko Widodo telah mem…"
85,"""tempo""","""HNW Mendukung Jamaah Umroh Fir…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Wakil Ketua MP…","""2023-03-04 06:18:04+00""","""[0.00841488,-0.023665192,0.006…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Wakil Ketua MPR RI Dr. H. M. H…"
86,"""tempo""","""Tim Dokkes Polri Telah Terima …","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Tim Kedokt…","""2023-03-04 06:44:10+00""","""[-0.012671886,-0.0039057182,0.…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Tim Kedokteran dan Kesehatan (…"
87,"""tempo""","""Bamsoet Ajak Komunitas Otomoti…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL - Ketua MPR RI s…","""2023-03-04 06:38:57+00""","""[-0.015486176,-0.0125719,-0.01…","""2023-03-04 07:03:39.039332""","""2023-03-04 07:03:39.039332""","""Ketua MPR RI Bambang Soesatyo …"
92,"""tempo""","""Korban Tewas Kebakaran Depo Pl…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta - Wakil Pres…","""2023-03-04 09:40:00+00""","""[-0.01298687,-0.024957022,0.01…","""2023-03-04 10:23:42.139686""","""2023-03-04 10:23:42.139686""","""Wakil Presiden Ma'ruf Amin men…"
125,"""tempo""","""14 Tahun Berkarya, PT SMI Siap…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""INFO NASIONAL Memperingati 14…","""2023-03-05 05:35:00+00""","""[-0.0021119912,-0.029577877,0.…","""2023-03-05 05:50:16.113131""","""2023-03-05 05:50:16.113131""","""Menteri Keuangan Sri Mulyani I…"
88,"""tempo""","""Penundaan Pemilu 2024 Bisa Bua…","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta -Wacana Pres…","""2023-03-04 07:36:57+00""","""[-0.011819591,-0.028945936,0.0…","""2023-03-04 08:04:06.161387""","""2023-03-04 08:04:06.161387""","""Wacana Presiden Joko Widodo at…"
93,"""tempo""","""Mabes Polri Selidiki Penyebab …","""https://statik.tempo.co/data/2…","""https://nasional.tempo.co/read…","""TEMPO.CO, Jakarta -Markas Besa…","""2023-03-04 09:06:44+00""","""[-0.008433584,-0.016826976,0.0…","""2023-03-04 10:23:42.139686""","""2023-03-04 10:23:42.139686""","""Markas Besar Kepolisian Republ…"


## Profil DuckDB

DuckDB membaca file langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [4]:
connection = duckdb.connect()
query = 'SUMMARIZE SELECT * FROM read_csv_auto(?)' if DATA_PATH.suffix == '.csv' else 'SUMMARIZE SELECT * FROM read_parquet(?)'
profil_duckdb = connection.execute(
    query, [str(DATA_PATH)]
).pl()
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""33""","""63567""",32684,"""34081.74877042921""","""18044.06863841403""","""17822""","""35101""","""49718""",32735,0.00
"""source""","""VARCHAR""","""cnbcindonesia""","""tempo""",6,null,null,null,null,null,32735,0.00
"""title""","""VARCHAR""",""" Daftar Panglima Kostrad Dari …","""​​Tips Atasi Masalah Jerawat d…",30501,null,null,null,null,null,32735,0.00
"""image""","""VARCHAR""","""https://akcdn.detik.net.id/vis…","""https://statik.tempo.co/data/2…",35667,null,null,null,null,null,32735,0.49
"""url""","""VARCHAR""","""https://bisnis.tempo.co/read/1…","""https://www.suara.com/news/202…",35724,null,null,null,null,null,32735,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""date""","""TIMESTAMP WITH TIME ZONE""","""2018-08-10 12:11:58+07""","""2023-04-12 03:13:25+07""",32566,"""2023-03-26 18:28:26.845792+07""",null,"""2023-03-18 02:47:05.723103+07""","""2023-03-26 09:50:07.279689+07""","""2023-04-04 13:55:59.920724+07""",32735,0.00
"""embedding""","""VARCHAR""","""[-0.000100255646,-0.00392203,0…","""[9.975993e-05,-0.027434079,0.0…",33604,null,null,null,null,null,32735,0.00
"""created_at""","""TIMESTAMP""","""2023-03-03 16:08:03.144472""","""2023-04-11 20:30:13.315332""",4156,"""2023-03-27 01:04:27.096193""",null,"""2023-03-18 08:07:56.954303""","""2023-03-27 00:12:04.55642""","""2023-04-04 19:58:22.910026""",32735,0.00


## Catatan Milestone 1

Berdasarkan hasil profiling awal menggunakan Polars LazyFrame dan DuckDB pada dataset `data.csv`, diperoleh temuan sebagai berikut:

### 1. Karakteristik & Dimensi Data
- **Nama File & Ukuran:** `data.csv` dengan ukuran fisik **737.9 MB** (737.915.204 bytes), memenuhi syarat minimum tugas (>= 500 MB).
- **Jumlah Baris:** **32.735 baris**.
- **Jumlah Kolom:** **11 kolom**.
- **Sumber Data:** Indonesian News Dataset (Hugging Face - `iqballx/indonesian_news_datasets`).

### 2. Tipe Kolom & Karakteristik Data
1. `id` (`Int64` / `BIGINT`): ID unik artikel berita (32.735 nilai unik, tidak ada duplikat ID).
2. `source` (`String` / `VARCHAR`): Media/portal penerbit berita (mencakup 7 portal berita).
3. `title` (`String` / `VARCHAR`): Judul artikel berita (32.651 nilai unik; terdapat 84 potensi judul berita ganda).
4. `image` (`String` / `VARCHAR`): Tautan gambar thumbnail (terdapat 160 baris bernilai string kosong `""`).
5. `url` (`String` / `VARCHAR`): Tautan web artikel berita (32.735 nilai unik).
6. `content` (`String` / `VARCHAR`): Teks isi berita lengkap berbahasa Indonesia.
7. `date` (`String` / `TIMESTAMP`): Waktu publikasi artikel berita.
8. `embedding` (`String` / `VARCHAR`): Vektor representasi semantik teks berdimensi tinggi (~768 dimensi), menyumbang proporsi ukuran file terbesar.
9. `created_at` & `updated_at` (`String` / `TIMESTAMP`): Timestamp metadata sistem.
10. `summary` (`String` / `VARCHAR`): Ringkasan artikel berita.

### 3. Kualitas Data & Missing Values
- **Kolom dengan Null Terbanyak:**
  - Kolom `summary` memiliki **10 missing values (`null`)** (~0.03%).
  - Kolom `image` memiliki **160 data string kosong (`""`)** (~0.49%).
- Seluruh kolom lainnya (`id`, `source`, `title`, `url`, `content`, `date`, `embedding`, `created_at`, `updated_at`) terisi lengkap tanpa missing value (0 null).

### 4. Anomali & Rekomendasi Penanganan untuk Milestone 2 (Data Cleaning)
1. **Parsing Tipe Data Waktu:** Kolom `date`, `created_at`, dan `updated_at` perlu diparsing secara konsisten ke tipe data Temporal (`Datetime`) untuk analisis tren waktu pada Milestone 3.
2. **Imputasi Nilai Null pada `summary`:** 10 baris missing values pada `summary` dapat diisi dengan cuplikan paragraf awal kolom `content` atau label representatif.
3. **Standarisasi String Kosong:** Kolom `image` yang berisi string kosong `""` sebaiknya distandarisasi menjadi `null` agar konsisten.
4. **Optimasi dan Konversi Format:** Menyimpan hasil akhir pembersihan data ke format Apache Parquet (`data/processed/dataset_bersih.parquet`) untuk kompresi ukuran data dan kecepatan query tinggi pada tahapan EDA.